# Lab 11A: Agentic AI for Audio
### AIML ZG527 — Audio Analysis 

**Authored by**: Seetha Parameswaran, BITS Pilani WILP

An **agentic audio system** does not just transcribe or classify — it *perceives*,
*reasons*, and *acts* in a loop to complete a user's goal. This lab builds a working
voice agent end to end and then **evaluates** it, mirroring the two ideas the industry
is converging on: **tool use** and **evals**.

**The Perception → Reasoning → Action loop demonstrated here:**

| Stage | Component | Tool used |
|-------|-----------|-----------|
| Perception | Speech → text | Whisper (ASR) |
| Reasoning | Intent + tool selection, with memory | LLM function calling |
| Action | Execute tool, then text → speech | Python functions + gTTS |

**What you will see:**
- A tool defined as a *function + JSON schema* the LLM can call (`get_weather`, `play_music`, `set_reminder`)
- The agent choosing *which* tool to call, executing it, and speaking the result
- Conversation **memory** carried across turns
- An **evaluation** block scoring the agent at the component level (ASR WER, intent,
  tool-call correctness) and the task level (end-to-end success), with an error-analysis trace

**Run modes** (choose at the bottom):
1. Full audio agent — microphone + OpenAI API
2. Simple agent — microphone, no API (rule-based)
3. Text-only — no microphone, OpenAI API
4. Minimal offline — no microphone, no API

> This is an **instructor demo**. By the end of the course you will have built every
> component in this pipeline yourself — this lab shows how they compose into an agent.

In [2]:
# ============================================
# IMPORTS
# ============================================
import whisper
import sounddevice as sd
import soundfile as sf
import numpy as np
from gtts import gTTS
from pydub import AudioSegment
from pydub.playback import play
import os
from openai import OpenAI

In [3]:
# ============================================
# AUDIO PERCEPTION: ASR with Whisper
# ============================================
def record_audio(duration=5, sample_rate=16000):
    """Record audio from microphone"""
    print(f"Recording for {duration} seconds...")
    audio = sd.rec(int(duration * sample_rate), 
                   samplerate=sample_rate, 
                   channels=1, 
                   dtype='float32')
    sd.wait()
    print("Recording complete!")
    return audio, sample_rate

def transcribe_audio(audio_path):
    """Transcribe audio using Whisper"""
    model = whisper.load_model("base")
    result = model.transcribe(audio_path)
    return result["text"]

def speech_to_text(audio, sample_rate):
    """Convert recorded audio to text"""
    # Save temporary audio file
    temp_file = "temp_audio.wav"
    sf.write(temp_file, audio, sample_rate)
    
    # Transcribe
    text = transcribe_audio(temp_file)
    
    # Cleanup
    os.remove(temp_file)
    
    return text

In [4]:
# ============================================
# AUDIO ACTION: TTS with gTTS
# ============================================
def text_to_speech(text):
    """Convert text to speech and play"""
    print(f"Agent says: {text}")
    
    # Generate speech
    tts = gTTS(text=text, lang='en', slow=False)
    tts.save("response.mp3")
    
    # Play audio
    audio = AudioSegment.from_mp3("response.mp3")
    play(audio)
    
    # Cleanup
    os.remove("response.mp3")

In [5]:
# ============================================
# REASONING: OpenAI GPT with Function Calling
# ============================================

# Define tools/functions the agent can use
def get_weather(location):
    """Get weather information (mock)"""
    return f"The weather in {location} is sunny and 25°C"

def play_music(genre):
    """Play music (mock)"""
    return f"Now playing {genre} music"

def set_reminder(task):
    """Set reminder (mock)"""
    return f"Reminder set: {task}"

In [6]:
# Tool definitions for OpenAI function calling
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get current weather for a location",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city name, e.g., San Francisco"
                    }
                },
                "required": ["location"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "play_music",
            "description": "Play music of a specific genre",
            "parameters": {
                "type": "object",
                "properties": {
                    "genre": {
                        "type": "string",
                        "description": "Music genre, e.g., jazz, rock, classical"
                    }
                },
                "required": ["genre"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "set_reminder",
            "description": "Set a reminder for a task",
            "parameters": {
                "type": "object",
                "properties": {
                    "task": {
                        "type": "string",
                        "description": "The reminder text"
                    }
                },
                "required": ["task"]
            }
        }
    }
]

# Available functions mapping
available_functions = {
    "get_weather": get_weather,
    "play_music": play_music,
    "set_reminder": set_reminder
}

In [7]:
# ============================================
# AGENT WITH MEMORY
# ============================================
class AudioAgent:
    """Audio agent with conversation memory"""
    
    def __init__(self, api_key):
        self.client = OpenAI(api_key=api_key)
        self.conversation_history = [
            {"role": "system", "content": "You are a helpful voice assistant. You can help with weather, music, and reminders."}
        ]
    
    def process(self, user_text):
        """Process user input and generate response"""
        
        # Add user message to history
        self.conversation_history.append({"role": "user", "content": user_text})
        
        # Get response from GPT with function calling
        response = self.client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=self.conversation_history,
            tools=tools,
            tool_choice="auto"
        )
        
        response_message = response.choices[0].message
        tool_calls = response_message.tool_calls
        
        # Handle function calls
        if tool_calls:
            # Add assistant response to history
            self.conversation_history.append(response_message)
            
            # Execute each function call
            for tool_call in tool_calls:
                function_name = tool_call.function.name
                function_args = eval(tool_call.function.arguments)
                
                # Call the function
                function_response = available_functions[function_name](**function_args)
                
                # Add function response to history
                self.conversation_history.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": function_name,
                    "content": function_response
                })
            
            # Get final response after function execution
            final_response = self.client.chat.completions.create(
                model="gpt-3.5-turbo",
                messages=self.conversation_history
            )
            
            final_message = final_response.choices[0].message.content
        else:
            final_message = response_message.content
        
        # Add assistant response to history
        self.conversation_history.append({"role": "assistant", "content": final_message})
        
        return final_message

In [8]:
# ============================================
# MAIN AGENT LOOP
# ============================================
def run_audio_agent():
    """Main agent loop: Listen → Think → Speak"""
    
    print("=" * 50)
    print("AUDIO AGENT DEMO")
    print("=" * 50)
    
    # Get OpenAI API key
    api_key = os.environ.get("OPENAI_API_KEY")
    if not api_key:
        api_key = input("Enter your OpenAI API key: ")
        os.environ["OPENAI_API_KEY"] = api_key
    
    # Initialize agent
    agent = AudioAgent(api_key)
    
    while True:
        print("\n[Press Enter to speak, or 'q' to quit]")
        user_input = input()
        
        if user_input.lower() == 'q':
            print("Goodbye!")
            break
        
        # STEP 1: PERCEPTION - Listen
        audio, sr = record_audio(duration=5)
        user_text = speech_to_text(audio, sr)
        print(f"You said: {user_text}")
        
        if not user_text.strip():
            print("No speech detected. Try again.")
            continue
        
        # STEP 2: REASONING - Think
        response = agent.process(user_text)
        
        # STEP 3: ACTION - Speak
        text_to_speech(response)

In [9]:
# ============================================
# SIMPLE VERSION (No OpenAI API)
# ============================================
def simple_audio_agent():
    """Simplified version without OpenAI API"""
    
    print("Simple Audio Agent (No API Required)")
    print("Say something, and I'll respond!")
    
    # Load Whisper model
    whisper_model = whisper.load_model("base")
    
    while True:
        print("\n[Press Enter to speak, or 'q' to quit]")
        user_input = input()
        
        if user_input.lower() == 'q':
            break
        
        # Record audio
        audio, sr = record_audio(duration=5)
        
        # Save and transcribe
        sf.write("temp.wav", audio, sr)
        result = whisper_model.transcribe("temp.wav")
        user_text = result["text"]
        os.remove("temp.wav")
        
        print(f"You said: {user_text}")
        
        # Simple rule-based response
        response = generate_response(user_text)
        
        # Speak response
        text_to_speech(response)

def generate_response(user_text):
    """Simple rule-based responses"""
    text_lower = user_text.lower()
    
    if "weather" in text_lower:
        return "The weather today is sunny and pleasant."
    elif "music" in text_lower or "play" in text_lower:
        return "Playing your favorite music now."
    elif "hello" in text_lower or "hi" in text_lower:
        return "Hello! How can I help you today?"
    elif "time" in text_lower:
        from datetime import datetime
        now = datetime.now()
        return f"The current time is {now.strftime('%H:%M')}"
    elif "reminder" in text_lower:
        return "I'll set a reminder for you."
    else:
        return "I heard you. Let me think about that."

In [10]:
# ============================================
# TEXT-ONLY VERSION (For Testing)
# ============================================
def text_only_agent():
    """Text-only version for testing without audio hardware"""
    
    print("Text-Only Audio Agent (for testing)")
    print("Type your message instead of speaking")
    
    # Get API key
    api_key = os.environ.get("OPENAI_API_KEY")
    if not api_key:
        api_key = input("Enter your OpenAI API key: ")
        os.environ["OPENAI_API_KEY"] = api_key
    
    agent = AudioAgent(api_key)
    
    while True:
        user_text = input("\nYou: ")
        
        if user_text.lower() in ['quit', 'exit', 'q']:
            print("Goodbye!")
            break
        
        # Get agent response
        response = agent.process(user_text)
        
        print(f"Agent: {response}")

In [11]:
# ============================================
# MINIMAL VERSION (Offline - No API, No Audio)
# ============================================
def minimal_offline_agent():
    """Minimal offline version - no API, no audio recording"""
    
    print("Minimal Offline Agent")
    print("Type your message (simulating speech)")
    
    # Simple conversation memory
    memory = []
    
    while True:
        user_text = input("\nYou: ")
        
        if user_text.lower() in ['quit', 'exit', 'q']:
            print("Goodbye!")
            break
        
        memory.append(f"User: {user_text}")
        
        # Rule-based response with context
        text_lower = user_text.lower()
        
        if "weather" in text_lower:
            response = "The weather is sunny and 25 degrees Celsius."
        elif "music" in text_lower or "play" in text_lower:
            if "jazz" in text_lower:
                response = "Playing jazz music now. Enjoy the smooth sounds!"
            elif "rock" in text_lower:
                response = "Playing rock music now. Let's rock!"
            else:
                response = "Playing your favorite music now."
        elif "reminder" in text_lower:
            response = "Reminder has been set. I'll notify you."
        elif "hello" in text_lower or "hi" in text_lower:
            response = "Hello! How can I help you today?"
        elif "help" in text_lower:
            response = "I can help with weather, play music, and set reminders. Just ask!"
        elif len(memory) > 1 and "what did i say" in text_lower:
            response = f"You said: {memory[-2]}"
        else:
            response = f"I understand you said: {user_text}. How can I help with that?"
        
        memory.append(f"Agent: {response}")
        print(f"Agent: {response}")
        
        # Keep only last 10 exchanges
        if len(memory) > 20:
            memory = memory[-20:]

In [12]:
# ============================================
# EVALUATION: Component-level + Task-level
# ============================================
# pip install jiwer
import json

try:
    from jiwer import wer
    HAS_JIWER = True
except ImportError:
    HAS_JIWER = False
    print("jiwer not installed -> skipping WER (pip install jiwer)")

# ---- Test set -------------------------------------------------
# audio_path + reference are optional; if omitted, ASR is skipped
# and 'query' is used directly as the (assumed-perfect) transcript.
test_cases = [
    {"query": "what's the weather in Mumbai",
     "expected_tool": "get_weather",
     "audio_path": None, "reference": None},
    {"query": "play some jazz",
     "expected_tool": "play_music",
     "audio_path": None, "reference": None},
    {"query": "remind me to call Anita at 5",
     "expected_tool": "set_reminder",
     "audio_path": None, "reference": None},
    {"query": "hello there",
     "expected_tool": None,            # no tool should be called
     "audio_path": None, "reference": None},
]

# ---- Component 1: Perception (ASR WER) ------------------------
def eval_asr(agent_client, audio_path, reference):
    if not (audio_path and reference and HAS_JIWER):
        return None, None            # skipped
    hyp = transcribe_audio(audio_path)
    return hyp, wer(reference, hyp)

# ---- Component 2+3: Reasoning -> tool selection ---------------
def predict_tool(client, text):
    """Return (tool_name, args) the model chooses, or (None, None)."""
    resp = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": "You are a helpful voice assistant."},
            {"role": "user", "content": text},
        ],
        tools=tools,
        tool_choice="auto",
    )
    tc = resp.choices[0].message.tool_calls
    if not tc:
        return None, None
    return tc[0].function.name, json.loads(tc[0].function.arguments)

# ---- Run evaluation ------------------------------------------
def run_evaluation(client):
    n = len(test_cases)
    intent_ok = tool_ok = task_ok = 0
    wers = []

    print("=" * 60)
    print("AGENT EVALUATION")
    print("=" * 60)

    for i, case in enumerate(test_cases, 1):
        # --- Perception ---
        hyp, w = eval_asr(client, case["audio_path"], case["reference"])
        transcript = hyp if hyp is not None else case["query"]
        if w is not None:
            wers.append(w)

        # --- Reasoning / tool call ---
        pred_tool, pred_args = predict_tool(client, transcript)

        # --- Component scores ---
        # intent = "did we correctly decide tool vs no-tool?"
        want_tool = case["expected_tool"] is not None
        got_tool = pred_tool is not None
        intent_correct = (want_tool == got_tool)
        tool_correct = (pred_tool == case["expected_tool"])
        task_pass = intent_correct and tool_correct

        intent_ok += intent_correct
        tool_ok += tool_correct
        task_ok += task_pass

        # --- Error-analysis trace ---
        print(f"\n[{i}] query: {case['query']!r}")
        if hyp is not None:
            mark = "OK" if w == 0 else f"WER={w:.2f}"
            print(f"    ASR      : {transcript!r}  [{mark}]")
        print(f"    intent   : {'OK' if intent_correct else 'FAIL'} "
              f"(want_tool={want_tool}, got_tool={got_tool})")
        print(f"    tool call: expected={case['expected_tool']} "
              f"got={pred_tool} args={pred_args}  "
              f"[{'OK' if tool_correct else 'FAIL'}]")
        print(f"    TASK     : {'PASS' if task_pass else 'FAIL'}")
        if not task_pass:
            stage = "ASR" if (hyp is not None and w) else \
                    "intent" if not intent_correct else "tool selection"
            print(f"    >> first failing stage: {stage}")

    # ---- Aggregate ----
    print("\n" + "=" * 60)
    print("SUMMARY (task-level + component-level)")
    print("=" * 60)
    if wers:
        print(f"  Mean WER              : {sum(wers)/len(wers):.3f}")
    print(f"  Intent accuracy       : {intent_ok}/{n} = {intent_ok/n:.0%}")
    print(f"  Tool-call correctness : {tool_ok}/{n} = {tool_ok/n:.0%}")
    print(f"  End-to-end task success: {task_ok}/{n} = {task_ok/n:.0%}")



In [13]:
# ============================================
# RUN DEMO
# ============================================

print("Choose demo mode:")
print("1. Full Audio Agent (requires microphone + OpenAI API)")
print("2. Simple Audio Agent (requires microphone, no API)")
print("3. Text-Only Agent (no microphone, requires OpenAI API)")
print("4. Minimal Offline Agent (no API, no microphone)")

choice = input("Enter choice (1/2/3/4): ")

if choice == "1":
    run_audio_agent()
elif choice == "2":
    simple_audio_agent()
elif choice == "3":
    text_only_agent()
elif choice == "4":
    minimal_offline_agent()
else:
    print("Invalid choice!")

Choose demo mode:
1. Full Audio Agent (requires microphone + OpenAI API)
2. Simple Audio Agent (requires microphone, no API)
3. Text-Only Agent (no microphone, requires OpenAI API)
4. Minimal Offline Agent (no API, no microphone)
Enter choice (1/2/3/4): 2
Simple Audio Agent (No API Required)
Say something, and I'll respond!

[Press Enter to speak, or 'q' to quit]

Recording for 5 seconds...
Recording complete!


/home/seetha/PythonScriptsCourses/PythonScriptsCourses/lib/python3.8/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


You said:  What is the weather today?
Agent says: The weather today is sunny and pleasant.


Input #0, wav, from '/tmp/tmpc59417pz.wav':   0KB sq=    0B f=0/0   
  Duration: 00:00:02.74, bitrate: 384 kb/s
    Stream #0:0: Audio: pcm_s16le ([1][0][0][0] / 0x0001), 24000 Hz, 1 channels, s16, 384 kb/s




[Press Enter to speak, or 'q' to quit]

Recording for 5 seconds...
Recording complete!


/home/seetha/PythonScriptsCourses/PythonScriptsCourses/lib/python3.8/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


You said:  Which location is this based on? Bangular or Moon?
Agent says: Hello! How can I help you today?


Input #0, wav, from '/tmp/tmpfefwng61.wav':   0KB sq=    0B f=0/0   
  Duration: 00:00:02.95, bitrate: 384 kb/s
    Stream #0:0: Audio: pcm_s16le ([1][0][0][0] / 0x0001), 24000 Hz, 1 channels, s16, 384 kb/s




[Press Enter to speak, or 'q' to quit]
q


In [ ]:
# Example:
# agent = AudioAgent(os.environ["OPENAI_API_KEY"])
# run_evaluation(agent.client)

## Conclusion

In this lab we assembled a complete **agentic audio system** and closed the loop with evaluation.

**What we built:**
- **Perception** — Whisper turned speech into text
- **Reasoning** — an LLM selected and called tools, guided by conversation memory
- **Action** — the chosen tool executed, and gTTS spoke the response
- **Tools** — each capability was a function exposed to the LLM via a JSON schema
- **Evaluation** — we measured the agent component by component (WER, intent accuracy,
  tool-call correctness) and end to end (task success), and traced *where* failures originate

**Why tools and evals matter:**
An agent is only as reliable as its weakest stage. A demo that "works once" is not a system —
tool-call correctness and a repeatable eval harness are what let you trust an agent with a
real user's request.

**Limitations of this demo (and what production systems add):**
- Mock tools — real agents call live APIs, with error handling and retries
- Single tool call per turn — production agents plan and chain multiple steps
- No safety layer — real deployments add confirmation before consequential actions

### Think about!!!
- How much should an audio agent do **without** confirming with the user?
- Your agent passes 9 of 10 test cases — is that good enough to ship? What would you measure first?
- Which audio actions (booking, buying, sending) should *always* require human approval?